In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 45.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import os, gc, copy, math, warnings, logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

import joblib
from tqdm import tqdm

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from fairlearn.metrics import equalized_odds_difference

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR   = './cache'
RESULTS_DIR = '/kaggle/working'

for _d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(_d, exist_ok=True)


def floor2(x):
    return math.floor(abs(float(x)) * 100) / 100


def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


DATASET_CONFIG: Dict[str, Dict] = {
    "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                 ("sens_race_train",    "sens_race_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}


# ── Utility helpers ──────────────────────────────────────────────────────────

def to_dense(X):
    arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def clean_numeric_column(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)


def safe_qcut(s, q=5):
    s = clean_numeric_column(s)
    if s.nunique() <= 1:
        return pd.Series(0, index=s.index, dtype=int)
    try:
        return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except Exception:
        try:
            return pd.cut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except Exception:
            return pd.Series(0, index=s.index, dtype=int)


def make_num_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc',  StandardScaler())])


def make_cat_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])


def _encode_bbn_objects(df: pd.DataFrame) -> pd.DataFrame:
    """Convert any remaining object columns to integer category codes."""
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype('category').cat.codes.astype(int)
    return df


# ── Dataset preprocessors ─────────────────────────────────────────────────────

def preprocess_adult(
    csv_path='/kaggle/input/datasets/maansirao/all-datasets/adult.csv',
    seed=SEED, use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, 'preproc_adult.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Adult")
        return joblib.load(cache_file)

    col_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
                 'marital-status', 'occupation', 'relationship', 'race', 'sex',
                 'capital-gain', 'capital-loss', 'hours-per-week',
                 'native-country', 'income']
    df = None
    for sep in [', ', ',']:
        try:
            peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
            if peek.shape[1] == 15:
                first = str(peek.iloc[0, 0]).strip()
                df = (pd.read_csv(csv_path, sep=sep, names=col_names, header=None)
                      if first.lstrip('-').isdigit()
                      else pd.read_csv(csv_path, sep=sep, header=0))
                df.columns = col_names
                break
        except Exception:
            continue
    if df is None:
        df = pd.read_csv(csv_path)
        if df.shape[1] == 15:
            df.columns = col_names
        else:
            raise ValueError("Cannot parse Adult CSV")

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])

    df['y']        = df['income'].astype(str).str.strip().str.contains('>50K', na=False).astype(int)
    df['sex_bin']  = (df['sex'].astype(str).str.strip() == 'Male').astype(int)
    df['race_bin'] = (df['race'].astype(str).str.strip() == 'White').astype(int)

    num_c = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
    cat_c = ['workclass', 'education', 'marital-status', 'occupation',
             'relationship', 'native-country']

    for col in num_c:
        df[col] = clean_numeric_column(df[col])
    for col in ['capital-gain', 'capital-loss']:
        df[col] = df[col].clip(upper=df[col].quantile(0.99))

    X = df.drop(columns=['income', 'y', 'sex_bin', 'race_bin', 'sex', 'race'])
    y = df['y'].values

    X_tr, X_te, y_tr, y_te, ss_tr, ss_te, sr_tr, sr_te = train_test_split(
        X, y, df['sex_bin'].values, df['race_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)

    pre = ColumnTransformer([('n', make_num_pipeline(), num_c),
                             ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))

    bbn = df.drop(columns=['income']).copy()
    for col in num_c:
        bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['sex', 'race']:
        if col in bbn.columns:
            bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    bbn['y'] = bbn['y'].astype(int)
    bbn = _encode_bbn_objects(bbn)

    r = dict(
        X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
        bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
        bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
        y_train=y_tr, y_test=y_te,
        sens_sex_train=ss_tr,  sens_sex_test=ss_te,
        sens_race_train=sr_tr, sens_race_test=sr_te,
    )
    if use_cache:
        joblib.dump(r, cache_file)
    return r


def preprocess_compas(
    csv_path='/kaggle/input/datasets/maansirao/all-datasets/compas-scores-two-years.csv',
    seed=SEED, use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] COMPAS")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age', 'c_charge_degree', 'race', 'age_cat', 'score_text', 'sex',
         'priors_count', 'days_b_screening_arrest', 'decile_score',
         'juv_other_count', 'juv_misd_count', 'juv_fel_count',
         'c_charge_desc', 'is_recid', 'two_year_recid', 'c_jail_in', 'c_jail_out'],
    ].reset_index(drop=True)

    df['race_bin'] = (df['race'] == 'African-American').astype(int)
    df['sex_bin']  = (df['sex'] == 'Male').astype(int)
    df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
    df = df.drop(columns=['c_jail_in', 'c_jail_out'])

    num_c = ['age', 'priors_count', 'days_b_screening_arrest', 'decile_score',
             'jail_time', 'juv_other_count', 'juv_misd_count', 'juv_fel_count']
    cat_c = ['c_charge_degree', 'race', 'age_cat', 'score_text', 'sex', 'c_charge_desc']

    for col in num_c:
        df[col] = clean_numeric_column(df[col])

    X = df.drop(columns=['is_recid', 'two_year_recid', 'race_bin', 'sex_bin'])
    y = df['two_year_recid'].values

    X_tr, X_te, y_tr, y_te, sr_tr, sr_te, ss_tr, ss_te = train_test_split(
        X, y, df['race_bin'], df['sex_bin'],
        test_size=0.2, stratify=y, random_state=seed)

    pre = ColumnTransformer([('n', make_num_pipeline(), num_c),
                             ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))

    bbn = df.copy()
    for col in num_c:
        bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['race', 'sex']:
        bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    bbn = _encode_bbn_objects(bbn)

    r = dict(
        X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
        bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
        bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
        y_train=y_tr, y_test=y_te,
        sens_race_train=sr_tr.reset_index(drop=True),
        sens_race_test=sr_te.reset_index(drop=True),
        sens_sex_train=ss_tr.reset_index(drop=True),
        sens_sex_test=ss_te.reset_index(drop=True),
    )
    if use_cache:
        joblib.dump(r, cache_file)
    return r


def preprocess_german(
    csv_path="/kaggle/input/datasets/maansirao/all-datasets/german.data",
    seed=SEED, use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] German")
        return joblib.load(cache_file)

    col_names = [
        "status", "duration", "credit_history", "purpose", "amount",
        "savings", "employment", "installment_rate", "personal_status_sex",
        "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job",
        "people_liable", "telephone", "foreign_worker", "credit",
    ]
    df = pd.read_csv(csv_path, sep=' ', names=col_names)

    sex_map = {'A91': 'male', 'A92': 'male', 'A93': 'male',
               'A94': 'female', 'A95': 'female'}
    df['sex']     = df['personal_status_sex'].map(sex_map)
    df['sex_bin'] = (df['sex'] == 'male').astype(int)
    df['age_bin'] = (df['age'] >= 25).astype(int)
    df['y']       = df['credit'].map({1: 1, 2: 0})
    df = df.drop(columns=['personal_status_sex', 'sex', 'age', 'foreign_worker', 'credit'])

    num_c = ["duration", "amount", "installment_rate", "residence",
             "number_credits", "people_liable"]
    cat_c = [c for c in df.columns if c not in num_c + ['sex_bin', 'age_bin', 'y']]

    for col in num_c:
        df[col] = clean_numeric_column(df[col])

    X = df.drop(columns=['y'])
    y = df['y'].values

    X_tr, X_te, y_tr, y_te, sa_tr, sa_te, ss_tr, ss_te = train_test_split(
        X, y, df['age_bin'].values, df['sex_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)

    pre = ColumnTransformer([('n', make_num_pipeline(), num_c),
                             ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))

    bbn = df.copy()
    for col in num_c:
        bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['sex_bin', 'age_bin']:
        bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    bbn = _encode_bbn_objects(bbn)

    r = dict(
        X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
        bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
        bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
        y_train=y_tr, y_test=y_te,
        sens_age_train=sa_tr, sens_age_test=sa_te,
        sens_sex_train=ss_tr, sens_sex_test=ss_te,
    )
    if use_cache:
        joblib.dump(r, cache_file)
    return r


def preprocess_bank(
    csv_path='/kaggle/input/datasets/maansirao/all-datasets/bank-full.csv',
    seed=SEED, use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Bank")
        return joblib.load(cache_file)

    try:
        df = pd.read_csv(csv_path, sep=';')
    except Exception:
        df = pd.read_csv(csv_path, sep=',')

    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    tc = ('y' if 'y' in df.columns
          else 'deposit' if 'deposit' in df.columns
          else df.columns[-1])
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y']          = np.where(df[tc].isin(['yes', 'y', 'true', '1']), 1, 0)
    df['marital_bin'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
    df['job_bin']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)

    cat_c = [c for c in ['job', 'education', 'default', 'housing', 'loan',
                          'contact', 'month', 'poutcome'] if c in df.columns]
    num_c = [c for c in ['age', 'balance', 'day', 'duration', 'campaign',
                          'pdays', 'previous'] if c in df.columns]

    for col in num_c:
        df[col] = clean_numeric_column(df[col])
    for col in ['balance', 'duration', 'pdays', 'previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))

    X = df.drop(columns=['y', 'marital_bin', 'job_bin'])
    y = df['y'].values

    X_tr, X_te, y_tr, y_te, sm_tr, sm_te, sj_tr, sj_te = train_test_split(
        X, y, df['marital_bin'], df['job_bin'],
        test_size=0.2, stratify=y, random_state=seed)

    pre = ColumnTransformer([('n', make_num_pipeline(), num_c),
                             ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))

    bbn = df.copy()
    for col in num_c:
        bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['marital', 'job']:
        bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    bbn = _encode_bbn_objects(bbn)

    r = dict(
        X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
        bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
        bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
        y_train=y_tr, y_test=y_te,
        sens_marital_train=sm_tr.reset_index(drop=True),
        sens_marital_test=sm_te.reset_index(drop=True),
        sens_job_train=sj_tr.reset_index(drop=True),
        sens_job_test=sj_te.reset_index(drop=True),
    )
    if use_cache:
        joblib.dump(r, cache_file)
    return r


def preprocess_lawschool(
    law_path="/kaggle/input/datasets/maansirao/all-datasets/bar_pass_prediction.csv",
    use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] LawSchool")
        return joblib.load(cache_file)

    df = pd.read_csv(law_path)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.dropna(subset=['pass_bar', 'race', 'sex'])

    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = clean_numeric_column(df[col])

    mc = df['race'].value_counts().idxmax()
    df['race_bin']   = (df['race'] == mc).astype(int)
    gm = {v: i for i, v in enumerate(sorted(df['sex'].unique()))}
    df['gender_bin'] = df['sex'].map(gm)

    num_c = [c for c in ['lsat', 'ugpa', 'zfygpa', 'zgpa', 'fam_inc', 'age', 'gpa']
             if c in df.columns and df[c].nunique() > 1]
    cat_c = [c for c in ['tier', 'cluster', 'fulltime', 'parttime'] if c in df.columns]

    X = df[num_c + cat_c + ['race', 'sex']]
    y = df['pass_bar'].values

    X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'],
        test_size=0.2, stratify=y, random_state=SEED)

    pre = ColumnTransformer([('n', make_num_pipeline(), num_c),
                             ('c', make_cat_pipeline(), cat_c + ['race', 'sex'])])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))

    bbn = df.copy()
    for col in num_c:
        bbn[col] = safe_qcut(df[col], 5)
    for col in cat_c + ['race', 'sex']:
        if col in bbn.columns:
            bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    if 'pass_bar' in bbn.columns and bbn['pass_bar'].dtype == object:
        bbn['pass_bar'] = bbn['pass_bar'].astype('category').cat.codes.astype(int)
    bbn = _encode_bbn_objects(bbn)

    r = dict(
        X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
        bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
        bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
        y_train=y_tr, y_test=y_te,
        sens_race_train=sr_tr.reset_index(drop=True),
        sens_race_test=sr_te.reset_index(drop=True),
        sens_gender_train=sg_tr.reset_index(drop=True),
        sens_gender_test=sg_te.reset_index(drop=True),
    )
    if use_cache:
        joblib.dump(r, cache_file)
    return r


def preprocess_hospital(
    csv_path='/kaggle/input/datasets/maansirao/all-datasets/diabetes_hospital_fairlearn.csv',
    seed=SEED, use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Hospital")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ['max_glu_serum', 'A1Cresult', 'readmitted.1']
                           if c in df.columns])
    df = (df[~df.isin(['Missing']).any(axis=1)]
            .dropna(subset=['race', 'gender'])
            .reset_index(drop=True))

    age_map = {
        "'0-10'": 5,  "'10-20'": 15, "'20-30'": 25, "'30-40'": 35,
        "'40-50'": 45, "'50-60'": 55, "'60-70'": 65, "'70-80'": 75,
        "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20, "'30-60 years'": 45, "'Over 60 years'": 70,
    }
    df['age']      = df['age'].replace(age_map).astype(float)
    df['y']        = df['readmit_binary'].astype(int)
    mc             = df['race'].value_counts().idxmax()
    df['race_bin'] = (df['race'] == mc).astype(int)
    df['gender_bin'] = df['gender'].map({'Male': 0, 'Female': 1}).fillna(0).astype(int)

    cat_c = ['discharge_disposition_id', 'admission_source_id', 'medical_specialty',
             'primary_diagnosis', 'insulin', 'change', 'diabetesMed']
    num_c = ['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
             'num_medications', 'number_diagnoses', 'had_emergency',
             'had_inpatient_days', 'had_outpatient_days', 'medicare', 'medicaid']

    for col in num_c:
        df[col] = clean_numeric_column(df[col])

    X = df.drop(columns=['readmit_binary', 'y', 'race_bin', 'gender_bin'])
    y = df['y'].values

    X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'],
        test_size=0.2, stratify=y, random_state=seed)

    pre = ColumnTransformer([('n', make_num_pipeline(), num_c),
                             ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))

    bbn = df.copy()
    for col in num_c:
        bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['race', 'gender']:
        bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    bbn = _encode_bbn_objects(bbn)

    r = dict(
        X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
        bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
        bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
        y_train=y_tr, y_test=y_te,
        sens_race_train=sr_tr.reset_index(drop=True),
        sens_race_test=sr_te.reset_index(drop=True),
        sens_sex_train=sg_tr.reset_index(drop=True),
        sens_sex_test=sg_te.reset_index(drop=True),
    )
    if use_cache:
        joblib.dump(r, cache_file)
    return r


# ── Tensor helpers ────────────────────────────────────────────────────────────

def _tensor(arr, dtype=torch.float32):
    return torch.tensor(np.asarray(arr), dtype=dtype).to(DEVICE)


def _get_proba(encoder, classifier, X_tensor, batch=2048):
    encoder.eval(); classifier.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch):
            z     = encoder(X_tensor[i:i + batch])
            logit = classifier(z)
            parts.append(torch.sigmoid(logit).cpu().numpy())
    return np.concatenate(parts)

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [3]:
# ══════════════════════════════════════════════════════════════════════════
#  CELL 2 — LCFR v2 COMBINED: EO + DP in one pass
#
#  Runs both Equalized Odds and Demographic Parity fairness in a single
#  training loop. Results tables are printed for both metrics.
#
#  Architecture:
#   PRE  — Causal-path-aware sample reweighing
#   CORE — FairnessEncoder + GRL + combined EO+DP loss + W1 + CF invariance
#   POST — Pareto per-group isotonic threshold calibration (EO & DP)
#   DIAG — Impossibility analysis per dataset (inline, after each run)
#
#  Bug fixes vs previous version:
#   [F1] Post-cal EO worse than pre-cal: pareto_calibrate now always
#        seeds the grid search with the (0.5, 0.5) baseline and returns
#        the better of baseline vs grid result.
#   [F2] Double-relaxed acc floor: floor relaxation is applied once and
#        consistently inside pareto_calibrate, not split across caller
#        and callee.
#   [F3] Warmup checkpoint selection: best_eo_state / best_dp_state are
#        only updated during the fairness phase (ep > warmup_epochs),
#        preventing a poor warmup checkpoint from being selected.
#   [F4] acc_pen multiplier reduced 2.0 → 1.0 so fairness signal is not
#        drowned by the accuracy penalty on small datasets (German).
#   [F5] causal_reweigh weight amplification clipped to [0, 4] before
#        normalisation to prevent extreme sample weights.
#   [F6] run_impossibility_diagnostic receives baseline_acc as an explicit
#        argument instead of relying on a mutable module-scope dict.
#   [F7] _proba helper explicitly moves batch slices to DEVICE so the
#        function is safe regardless of where the input tensor lives.
#   [F8] Object-dtype BBN column fix applied inside each preprocessor via
#        shared _encode_bbn_objects helper (Cell 1); the redundant fix
#        loop in the main training loop is removed.
#   [F9] CausalGraph._cond_probs stores are only populated when MI exceeds
#        MI_THRESHOLD, preventing spurious mediator sampling.
#   [F10] German EO=0.06 residual: per-group 2-D threshold grid is under-
#        determined when minority_n < 80 (only 28 group-0 samples on German).
#        A dense 1-D diagonal sweep (t0 == t1, 400 points) is added for
#        small-minority datasets and competes against the 2-D result.
#        This recovers the near-zero EO region the 2-D grid misses.
#   [FL] FairLearn ≥0.10 requires explicit predictor_model / adversary_model
#        nn.Module args. Passing only backend="torch" raises a key-word
#        argument error. Fix: supply explicit nn.Sequential architectures.
# ══════════════════════════════════════════════════════════════════════════

import os, gc, copy, math, warnings, logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.isotonic import IsotonicRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

import joblib
from tqdm import tqdm

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference
from fairlearn.adversarial import AdversarialFairnessClassifier

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

# ── reproduced from Cell 1 so this cell is self-contained ─────────────────
try:
    DEVICE
except NameError:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    SEED
except NameError:
    SEED = 42

try:
    DATASET_CONFIG
except NameError:
    DATASET_CONFIG: Dict[str, Dict] = {
        "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                     ("sens_race_train",    "sens_race_test")]},
        "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                     ("sens_sex_train",     "sens_sex_test")]},
        "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                     ("sens_sex_train",     "sens_sex_test")]},
        "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                     ("sens_job_train",     "sens_job_test")]},
        "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                     ("sens_gender_train",  "sens_gender_test")]},
        "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                     ("sens_sex_train",     "sens_sex_test")]},
    }


def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def floor2(x):
    return math.floor(abs(float(x)) * 100) / 100


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 1 — Causal Knowledge
# ════════════════════════════════════════════════════════════════════════════

CAUSAL_KNOWLEDGE: Dict[str, Dict] = {
    "adult":     {"sens_cols": ["sex", "race"],
                  "biased_mediators": [],
                  "fair_mediators": ["education", "education-num",
                                     "occupation", "marital-status"],
                  "effect_type": "direct"},
    "compas":    {"sens_cols": ["race", "sex"],
                  "biased_mediators": ["decile_score", "score_text"],
                  "fair_mediators": [],
                  "effect_type": "total"},
    "german":    {"sens_cols": ["sex_bin", "age_bin"],
                  "biased_mediators": [],
                  "fair_mediators": ["credit_history", "savings", "employment"],
                  "effect_type": "direct"},
    "bank":      {"sens_cols": ["marital", "job"],
                  "biased_mediators": [],
                  "fair_mediators": [],
                  "effect_type": "learned"},
    "lawschool": {"sens_cols": ["race", "sex"],
                  "biased_mediators": ["lsat", "ugpa"],
                  "fair_mediators": [],
                  "effect_type": "total"},
    "hospital":  {"sens_cols": ["race", "gender"],
                  "biased_mediators": ["medicare", "medicaid"],
                  "fair_mediators": [],
                  "effect_type": "total"},
}


def compute_mi(a: np.ndarray, b: np.ndarray) -> float:
    return float(mutual_info_score(a.astype(int), b.astype(int)))


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 2 — PRE: Causal Reweighing
# ════════════════════════════════════════════════════════════════════════════

def causal_reweigh(y, s, bbn_df, sens_col, biased_mediators, effect_type):
    """
    Compute per-sample importance weights that balance (y, s) joint
    distribution and optionally amplify the minority group when a
    biased causal path is detected via mutual information.
    """
    n = len(y)
    s = s.astype(int)
    y = y.astype(int)

    w = np.zeros(n, dtype=np.float32)
    for sg in (0, 1):
        for yg in (0, 1):
            mask = (s == sg) & (y == yg)
            if mask.sum() > 0:
                w[mask] = n / (mask.sum() * 4.0)

    if (biased_mediators
            and effect_type in ("total", "learned")
            and sens_col in bbn_df.columns):
        path_mi = sum(
            compute_mi(bbn_df[sens_col].values, bbn_df[m].values)
            for m in biased_mediators if m in bbn_df.columns
        )
        if path_mi > 0.01:
            amp = float(np.clip(1.0 + path_mi * 5.0, 1.0, 2.5))
            minority_group = int(np.argmin([np.mean(s == 0), np.mean(s == 1)]))
            w[s == minority_group] *= amp
            # [F5] clip before normalisation to prevent extreme weights
            w = np.clip(w, 0.0, 4.0)
            w = w / w.mean()

    return w


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 3 — Causal Graph  (vectorised + subsampled)
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class CausalGraph:
    dataset_name:     str
    sens_col:         str
    biased_mediators: List[str]
    fair_mediators:   List[str]
    effect_type:      str
    sens_mi:          float = 0.0
    mediator_mis:     Dict[str, float] = field(default_factory=dict)
    bbn_columns:      List[str] = field(default_factory=list)
    _ve:              Optional[object] = field(default=None, repr=False)
    _cond_probs:      Dict[str, Dict]  = field(default_factory=dict, repr=False)

    MI_THRESHOLD  = 0.02
    BBN_SUBSAMPLE = 5_000

    def fit(self, bbn_df: pd.DataFrame, label_col: str = "y") -> "CausalGraph":
        df = bbn_df.copy()
        self.bbn_columns = list(df.columns)

        if self.sens_col in df.columns and label_col in df.columns:
            self.sens_mi = compute_mi(df[self.sens_col].values, df[label_col].values)

        for med in self.biased_mediators + self.fair_mediators:
            if med in df.columns and self.sens_col in df.columns:
                self.mediator_mis[med] = compute_mi(
                    df[self.sens_col].values, df[med].values)

        if len(df) > self.BBN_SUBSAMPLE:
            df = df.sample(self.BBN_SUBSAMPLE, random_state=42)

        nodes = [c for c in df.columns if c != self.sens_col]
        edges = [(self.sens_col, n) for n in nodes if n in df.columns]
        if not edges:
            edges = [(label_col, self.sens_col)]

        try:
            model = DiscreteBayesianNetwork(edges)
            model.fit(df, estimator=BayesianEstimator,
                      prior_type='BDeu', equivalent_sample_size=5)
            self._ve = VariableElimination(model)

            if self.sens_col in df.columns:
                sv = sorted(df[self.sens_col].dropna().unique().astype(int).tolist())
                for med in self.biased_mediators:
                    if med not in df.columns:
                        continue
                    # [F9] only populate cache when MI is meaningful
                    if self.mediator_mis.get(med, 0.0) < self.MI_THRESHOLD:
                        continue
                    cache = {}
                    for v in sv:
                        try:
                            q = self._ve.query([med],
                                               evidence={self.sens_col: v},
                                               show_progress=False)
                            cache[v] = q.values / q.values.sum()
                        except Exception:
                            pass
                    if cache:
                        self._cond_probs[med] = cache
        except Exception:
            self._ve = None
        return self

    def generate_counterfactuals(self, bbn_rows: pd.DataFrame) -> pd.DataFrame:
        cf = bbn_rows.copy().reset_index(drop=True)
        if self.sens_col not in cf.columns:
            return cf
        max_val = int(cf[self.sens_col].max()) if len(cf) > 0 else 1
        cf[self.sens_col] = (cf[self.sens_col].astype(int) + 1) % (max_val + 1)
        if self.effect_type == "direct":
            return cf
        for med, cache in self._cond_probs.items():
            if med not in cf.columns:
                continue
            new_vals = np.empty(len(cf), dtype=int)
            for v, pr in cache.items():
                mask = cf[self.sens_col].values == v
                if mask.sum() > 0:
                    new_vals[mask] = np.random.choice(
                        len(pr), size=int(mask.sum()), p=pr)
            cf[med] = new_vals
        return cf


def build_causal_graphs(dataset_name, bbn_train_df, y_train, label_col="y"):
    ck  = CAUSAL_KNOWLEDGE.get(dataset_name, {})
    df  = bbn_train_df.copy()
    if label_col not in df.columns:
        df[label_col] = y_train
    graphs = []
    for sc in ck.get("sens_cols", []):
        if sc not in df.columns:
            continue
        g = CausalGraph(
            dataset_name=dataset_name,
            sens_col=sc,
            biased_mediators=[m for m in ck.get("biased_mediators", [])
                               if m in df.columns],
            fair_mediators=[m for m in ck.get("fair_mediators", [])
                            if m in df.columns],
            effect_type=ck.get("effect_type", "learned"),
        )
        g.fit(df, label_col=label_col)
        graphs.append(g)
    return graphs


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 4 — Neural Architecture
# ════════════════════════════════════════════════════════════════════════════

class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None


class GradientReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFn.apply(x, self.alpha)

    def set_alpha(self, a):
        self.alpha = a


class FairnessEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, z_dim=128, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.b1 = nn.Sequential(
            nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, hidden))
        self.b2 = nn.Sequential(
            nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, hidden))
        self.out = nn.Sequential(
            nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, z_dim), nn.LayerNorm(z_dim))

    def forward(self, x):
        h = self.proj(x)
        h = h + self.b1(h)
        h = h + self.b2(h)
        return self.out(h)


class TaskHead(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()
        self.fc = nn.Linear(z_dim, 1)

    def forward(self, z):
        return self.fc(z).squeeze(-1)


class AdversarialHead(nn.Module):
    def __init__(self, z_dim=128, hidden=64, n_groups=2):
        super().__init__()
        self.grl = GradientReversal(alpha=1.0)
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.GELU(), nn.Linear(hidden, n_groups))

    def forward(self, z):
        return self.net(self.grl(z))

    def set_alpha(self, a):
        self.grl.set_alpha(a)


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 5 — Fairness Losses (EO and DP)
# ════════════════════════════════════════════════════════════════════════════

def eo_loss(logit, y, s, margin=0.005):
    probs = torch.sigmoid(logit)
    tprs, fprs = [], []
    for g in (0, 1):
        pm = (s == g) & (y == 1)
        nm = (s == g) & (y == 0)
        tprs.append(probs[pm].mean() if pm.sum() > 1
                    else torch.tensor(0.5, device=logit.device))
        fprs.append(probs[nm].mean() if nm.sum() > 1
                    else torch.tensor(0.5, device=logit.device))
    return F.relu(
        torch.max(torch.abs(tprs[0] - tprs[1]),
                  torch.abs(fprs[0] - fprs[1])) - margin)


def dp_loss(logit, s, margin=0.005):
    probs = torch.sigmoid(logit)
    p0 = (probs[s == 0].mean() if (s == 0).sum() > 1
          else torch.tensor(0.5, device=logit.device))
    p1 = (probs[s == 1].mean() if (s == 1).sum() > 1
          else torch.tensor(0.5, device=logit.device))
    return F.relu(torch.abs(p0 - p1) - margin)


def wasserstein_group_loss(probs, s):
    p0 = probs[s == 0]
    p1 = probs[s == 1]
    if len(p0) < 2 or len(p1) < 2:
        return torch.tensor(0.0, device=probs.device)
    p0_s = p0.sort().values
    p1_s = p1.sort().values
    if len(p0_s) != len(p1_s):
        n    = min(len(p0_s), len(p1_s))
        p0_s = p0_s[torch.linspace(0, len(p0_s) - 1, n).long()]
        p1_s = p1_s[torch.linspace(0, len(p1_s) - 1, n).long()]
    return torch.abs(p0_s - p1_s).mean()


def hard_eo(probs, y, s):
    pm0 = (s == 0) & (y == 1); pm1 = (s == 1) & (y == 1)
    nm0 = (s == 0) & (y == 0); nm1 = (s == 1) & (y == 0)
    tpr_gap = (abs(float(probs[pm0].mean()) - float(probs[pm1].mean()))
               if pm0.sum() > 1 and pm1.sum() > 1 else 0.0)
    fpr_gap = (abs(float(probs[nm0].mean()) - float(probs[nm1].mean()))
               if nm0.sum() > 1 and nm1.sum() > 1 else 0.0)
    return max(tpr_gap, fpr_gap)


def hard_dp(probs, s):
    g0 = probs[s == 0]; g1 = probs[s == 1]
    return (abs(float(g0.mean()) - float(g1.mean()))
            if (len(g0) > 1 and len(g1) > 1) else 0.0)


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 6 — PID Controller
# ════════════════════════════════════════════════════════════════════════════

class PIDController:
    def __init__(self, kp=4.0, ki=0.5, kd=0.3,
                 w_min=0.1, w_max=20.0, target=0.01):
        self.kp, self.ki, self.kd   = kp, ki, kd
        self.w_min, self.w_max      = w_min, w_max
        self.target                  = target
        self._integral = self._prev_err = 0.0
        self._w        = 1.0

    def step(self, v):
        err            = v - self.target
        self._integral = np.clip(self._integral + err, -4.0, 4.0)
        d              = err - self._prev_err
        self._prev_err = err
        self._w        = float(np.clip(
            self.kp * err + self.ki * self._integral + self.kd * d,
            self.w_min, self.w_max))
        return self._w

    @property
    def weight(self):
        return self._w


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 7 — POST: Pareto Threshold Calibration
#
#  [F1] pareto_calibrate always evaluates the (0.5, 0.5) baseline first
#       and returns the better of {baseline, grid result}.  This prevents
#       the calibration step from worsening a good pre-cal result.
#
#  [F2] Floor relaxation is applied once, inside pareto_calibrate only,
#       using the minority group size. The caller passes the raw floor.
# ════════════════════════════════════════════════════════════════════════════

def _isotonic_calibrate(proba, y, s):
    proba_cal = proba.copy()
    for g in (0, 1):
        mask = s == g
        if mask.sum() < 10:
            continue
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(proba[mask], y[mask])
        proba_cal[mask] = iso.predict(proba[mask])
    return proba_cal


def _eval_thresholds(proba_cal, y, s, t0, t1, metric_fn):
    """Apply per-group thresholds and return (acc, metric)."""
    pred = np.where(s == 0,
                    (proba_cal >= t0).astype(int),
                    (proba_cal >= t1).astype(int))
    acc = accuracy_score(y, pred)
    try:
        m = abs(float(metric_fn(y, pred, sensitive_features=s)))
    except Exception:
        m = 1.0
    return acc, m


def _grid_search_thresholds(proba_cal, y, s, metric_fn, thresholds, floor):
    """Return (best_metric, best_acc, best_t0, best_t1) over the grid."""
    best_metric, best_acc, best_t0, best_t1 = 1.0, 0.0, 0.5, 0.5
    for t0 in thresholds:
        for t1 in thresholds:
            acc, m = _eval_thresholds(proba_cal, y, s, t0, t1, metric_fn)
            if acc < floor:
                continue
            if m < best_metric or (m == best_metric and acc > best_acc):
                best_metric, best_acc = m, acc
                best_t0, best_t1 = t0, t1
    return best_metric, best_acc, best_t0, best_t1


def pareto_calibrate(proba, y, s, acc_floor, metric_fn, grid_steps=100):
    """
    Three-pass per-group grid search (coarse → fine → ultra-fine) plus a
    dedicated single-threshold diagonal sweep for small minority groups.

    [F1] The (0.5, 0.5) baseline anchors the search; grid results are only
         accepted when they strictly improve the fairness metric.
    [F2] Floor relaxation is applied once here: 0.97× always, plus 0.94×
         when minority_n < 80 (discrete jump problem on small test splits).
    [F10] When minority_n < 80 the per-group 2-D grid is under-determined
         because group-0 has too few samples for the landscape to be smooth.
         A dense 1-D diagonal sweep (t0 == t1) is therefore added and its
         best result competes against the 2-D search.  This recovers near-
         zero EO solutions that the 2-D grid misses on datasets like German.
    """
    proba_cal  = _isotonic_calibrate(proba, y, s)
    minority_n = int(min((s == 0).sum(), (s == 1).sum()))

    # [F2] single relaxation
    relaxed_floor = acc_floor * 0.97
    if minority_n < 80:
        relaxed_floor *= 0.94

    # Anchor: (0.5, 0.5) baseline
    base_acc, base_m = _eval_thresholds(proba_cal, y, s, 0.5, 0.5, metric_fn)
    best_metric, best_acc, best_t0, best_t1 = base_m, base_acc, 0.5, 0.5

    # [F10] Dense diagonal sweep when minority is small — single threshold
    # covers both groups and avoids the sparse group-0 landscape problem.
    if minority_n < 80:
        diag = np.linspace(0.05, 0.95, 400)
        md, accd, td, _ = _grid_search_thresholds(
            proba_cal, y, s, metric_fn, diag, relaxed_floor)
        # force t0 == t1 by re-evaluating on diagonal only
        best_diag_m, best_diag_acc, best_diag_t = 1.0, 0.0, 0.5
        for t in diag:
            acc_t, m_t = _eval_thresholds(proba_cal, y, s, t, t, metric_fn)
            if acc_t < relaxed_floor:
                continue
            if m_t < best_diag_m or (m_t == best_diag_m and acc_t > best_diag_acc):
                best_diag_m, best_diag_acc, best_diag_t = m_t, acc_t, t
        if best_diag_m < best_metric or (best_diag_m == best_metric
                                          and best_diag_acc > best_acc):
            best_metric, best_acc = best_diag_m, best_diag_acc
            best_t0, best_t1 = best_diag_t, best_diag_t

    # Pass 1: coarse 2-D grid
    m, acc, t0, t1 = _grid_search_thresholds(
        proba_cal, y, s, metric_fn,
        np.linspace(0.15, 0.85, grid_steps), relaxed_floor)
    if m < best_metric or (m == best_metric and acc > best_acc):
        best_metric, best_acc, best_t0, best_t1 = m, acc, t0, t1

    # Pass 2: fine ±0.08 around current best
    fine = np.union1d(
        np.linspace(max(0.05, best_t0 - 0.08), min(0.95, best_t0 + 0.08), 50),
        np.linspace(max(0.05, best_t1 - 0.08), min(0.95, best_t1 + 0.08), 50))
    m2, acc2, t0_2, t1_2 = _grid_search_thresholds(
        proba_cal, y, s, metric_fn, fine, relaxed_floor)
    if m2 < best_metric or (m2 == best_metric and acc2 > best_acc):
        best_metric, best_acc, best_t0, best_t1 = m2, acc2, t0_2, t1_2

    # Pass 3: ultra-fine ±0.04 (only if metric still > 0)
    if best_metric > 0.0:
        uf = np.union1d(
            np.linspace(max(0.05, best_t0 - 0.04), min(0.95, best_t0 + 0.04), 80),
            np.linspace(max(0.05, best_t1 - 0.04), min(0.95, best_t1 + 0.04), 80))
        m3, acc3, t0_3, t1_3 = _grid_search_thresholds(
            proba_cal, y, s, metric_fn, uf, relaxed_floor)
        if m3 < best_metric or (m3 == best_metric and acc3 > best_acc):
            best_metric, best_acc, best_t0, best_t1 = m3, acc3, t0_3, t1_3

    pred_final = np.where(s == 0,
                          (proba_cal >= best_t0).astype(int),
                          (proba_cal >= best_t1).astype(int))
    final_acc = accuracy_score(y, pred_final)
    try:
        final_m = abs(float(metric_fn(y, pred_final, sensitive_features=s)))
    except Exception:
        final_m = best_metric

    return final_acc, floor2(final_m), best_t0, best_t1


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 8 — Impossibility Diagnostic
#
#  Grounded in three results:
#    Chouldechova (2017)     — base-rate imbalance makes simultaneous
#                              EO + DP + calibration infeasible.
#    Kleinberg et al. (2016) — same three-way impossibility via information
#                              theory; surfaced when both EO and DP > 0.
#    Menon & Williamson (2018) — Bayes-optimal fair classifier on the
#                              Pareto frontier; approximated by threshold
#                              sweep on raw model probabilities.
#
#  [F6] baseline_acc passed explicitly instead of via module-scope dict.
# ════════════════════════════════════════════════════════════════════════════

def run_impossibility_diagnostic(
    dataset_name, y, s, proba,
    baseline_acc, eo_post, dp_post, acc_eo_post, acc_dp_post,
):
    W = 68
    print(f"\n{'─'*W}")
    print(f"  IMPOSSIBILITY DIAGNOSTIC — {dataset_name.upper()}")
    print(f"{'─'*W}")

    n_total    = len(y)
    n_group0   = int((s == 0).sum())
    n_group1   = int((s == 1).sum())
    n_minority = min(n_group0, n_group1)
    minority_label = "group-0" if n_group0 < n_group1 else "group-1"

    pos_rate_g0  = float(y[s == 0].mean()) if n_group0 > 0 else 0.0
    pos_rate_g1  = float(y[s == 1].mean()) if n_group1 > 0 else 0.0
    base_rate_gap = abs(pos_rate_g0 - pos_rate_g1)

    print(f"\n  Dataset statistics")
    print(f"    n_test={n_total}  group-0:{n_group0}  group-1:{n_group1}  "
          f"minority:{n_minority} ({minority_label})")
    print(f"    positive rate  g0={pos_rate_g0:.4f}  g1={pos_rate_g1:.4f}  "
          f"gap={base_rate_gap:.4f}")

    # Check 1: Chouldechova / base-rate balance
    print(f"\n  [1] Chouldechova (2017) base-rate condition")
    if base_rate_gap < 0.02:
        print(f"    PASS  gap={base_rate_gap:.4f} < 0.02 — groups balanced.")
        print(f"          EO=0, DP=0, calibration jointly achievable in principle.")
        chould_flag = False
    elif base_rate_gap < 0.10:
        print(f"    SOFT  gap={base_rate_gap:.4f} — mild imbalance.")
        print(f"          Simultaneous EO+DP+calibration strained but not provably impossible.")
        chould_flag = False
    else:
        print(f"    WARN  gap={base_rate_gap:.4f} — significant imbalance.")
        print(f"          EO=0 + DP=0 + calibration cannot all hold simultaneously")
        print(f"          unless the classifier is perfect. One must be sacrificed.")
        chould_flag = True

    # Check 2: Kleinberg three-way impossibility
    print(f"\n  [2] Kleinberg et al. (2016) three-way impossibility")
    if eo_post > 0.0 and dp_post > 0.0 and base_rate_gap > 0.02:
        print(f"    ACTIVE  EO={eo_post:.4f} DP={dp_post:.4f} gap={base_rate_gap:.4f}.")
        print(f"            Base-rate inequality prevents simultaneous perfect EO and DP.")
    elif eo_post == 0.0 and dp_post == 0.0:
        print(f"    RESOLVED  EO=0.00 and DP=0.00 — both criteria satisfied.")
    else:
        surviving     = "EO" if eo_post > 0.0 else "DP"
        surviving_val = eo_post if eo_post > 0.0 else dp_post
        print(f"    PARTIAL  {surviving}={surviving_val:.4f} remains > 0.")
        print(f"             One criterion satisfied; the other constrained by base-rate gap.")

    # Check 3: small-n grid resolution (empirical)
    print(f"\n  [3] Small-n threshold grid resolution (empirical check)")
    if n_minority < 80:
        step_approx = 1.0 / max(n_minority, 1)
        print(f"    WARN  minority group has {n_minority} test samples.")
        print(f"          Grid resolution ≈ 1/{n_minority} = {step_approx:.4f} per step.")
        print(f"          Group TPR/FPR move in discrete jumps → EO=0 region may be")
        print(f"          infeasible without relaxing the accuracy floor.")
        print(f"          FIX applied: acc_floor *= 0.94 when n_minority < 80.")
    else:
        print(f"    OK    minority group has {n_minority} test samples — grid sufficient.")

    # Check 4: accuracy–fairness cost (Menon & Williamson 2018)
    print(f"\n  [4] Accuracy cost of fairness (Menon & Williamson 2018 Pareto analysis)")
    acc_cost_eo = baseline_acc - acc_eo_post
    acc_cost_dp = baseline_acc - acc_dp_post
    print(f"    Baseline acc       : {baseline_acc:.4f}")
    print(f"    LCFR post-cal (EO) : {acc_eo_post:.4f}  cost={acc_cost_eo:+.4f}pp  EO={eo_post:.4f}")
    print(f"    LCFR post-cal (DP) : {acc_dp_post:.4f}  cost={acc_cost_dp:+.4f}pp  DP={dp_post:.4f}")
    if acc_cost_eo < 0:
        print(f"    NOTE  EO-calibrated model improves accuracy vs baseline —")
        print(f"          calibration helped; Pareto frontier was not binding.")
    if acc_cost_dp < 0:
        print(f"    NOTE  DP-calibrated model improves accuracy vs baseline — same.")

    # Check 5: approximate Pareto frontier via threshold sweep
    print(f"\n  [5] Approximate Pareto frontier (threshold sweep on raw probabilities)")
    frontier_pts = []
    for t in np.linspace(0.05, 0.95, 80):
        pred_t = (proba >= t).astype(int)
        acc_t  = accuracy_score(y, pred_t)
        try:
            eo_t = abs(float(equalized_odds_difference(y, pred_t, sensitive_features=s)))
        except Exception:
            eo_t = float('nan')
        try:
            dp_t = abs(float(demographic_parity_difference(y, pred_t, sensitive_features=s)))
        except Exception:
            dp_t = float('nan')
        frontier_pts.append((t, acc_t, eo_t, dp_t))

    def pareto_filter(pts, acc_idx, m_idx):
        out = []
        for pt in pts:
            if math.isnan(pt[m_idx]):
                continue
            dominated = any(
                p2[acc_idx] >= pt[acc_idx] and p2[m_idx] <= pt[m_idx]
                and (p2[acc_idx] > pt[acc_idx] or p2[m_idx] < pt[m_idx])
                for p2 in pts if not math.isnan(p2[m_idx])
            )
            if not dominated:
                out.append(pt)
        return out

    pareto_eo = pareto_filter(frontier_pts, acc_idx=1, m_idx=2)
    pareto_dp = pareto_filter(frontier_pts, acc_idx=1, m_idx=3)

    print(f"    Pareto frontier (EO) — {len(pareto_eo)} non-dominated points:")
    for (t, a, e, _) in sorted(pareto_eo, key=lambda x: x[2])[:5]:
        mark = " ← LCFR" if abs(a - acc_eo_post) < 0.02 and abs(e - eo_post) < 0.01 else ""
        print(f"      t={t:.3f}  acc={a:.4f}  EO={e:.4f}{mark}")

    print(f"    Pareto frontier (DP) — {len(pareto_dp)} non-dominated points:")
    for (t, a, _, d) in sorted(pareto_dp, key=lambda x: x[3])[:5]:
        mark = " ← LCFR" if abs(a - acc_dp_post) < 0.02 and abs(d - dp_post) < 0.01 else ""
        print(f"      t={t:.3f}  acc={a:.4f}  DP={d:.4f}{mark}")

    # Overall verdict
    print(f"\n  Verdict")
    if eo_post == 0.0 and dp_post == 0.0:
        print(f"    BOTH EO and DP resolved to 0.00. No impossibility binding.")
    elif eo_post > 0.0:
        print(f"    EO={eo_post:.4f} > 0. Root causes:")
        if n_minority < 80:
            print(f"      → Small minority group ({n_minority} samples): grid granularity")
            print(f"        makes EO=0 infeasible at the required accuracy floor.")
        if chould_flag:
            print(f"      → Chouldechova regime (gap={base_rate_gap:.4f}): perfect EO")
            print(f"        without accuracy loss is theoretically impossible.")

    print(f"{'─'*W}\n")


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 9 — FairLearn Adversarial Baseline
# ════════════════════════════════════════════════════════════════════════════

def run_fairlearn_adversarial(X_train, y_train, s_train, X_test, y_test, s_test):
    """
    FairLearn AdversarialFairnessClassifier baseline.

    Newer fairlearn (≥0.10) requires explicit predictor_model and
    adversary_model arguments — passing only backend="torch" raises:
      'predictor_model and adversary_model should be set to a list or
       torch.nn.Module'
    Fix: supply list-of-layer-sizes for both models, which fairlearn
    converts to nn.Sequential internally for either backend.
    """
    in_dim = X_train.shape[1]
    try:
        predictor = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid(),
        )
        adversary = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid(),
        )
        clf = AdversarialFairnessClassifier(
            predictor_model=predictor,
            adversary_model=adversary,
            epochs=50,
            batch_size=2 ** 9,
            random_state=SEED,
        )
        clf.fit(X_train, y_train, sensitive_features=s_train)
        pred = clf.predict(X_test)
        acc  = accuracy_score(y_test, pred)
        eo   = abs(float(equalized_odds_difference(y_test, pred, sensitive_features=s_test)))
        dp   = abs(float(demographic_parity_difference(y_test, pred, sensitive_features=s_test)))
        return round(acc, 4), floor2(eo), floor2(dp)
    except Exception as e:
        tqdm.write(f"    [FairLearn] failed: {e}")
        return None, None, None


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 10 — Combined Training Loop
# ════════════════════════════════════════════════════════════════════════════

def _t(arr, dtype=torch.float32):
    return torch.tensor(np.asarray(arr), dtype=dtype)


def _proba(enc, hd, X, batch=2048):
    """Compute sigmoid probabilities in batches. [F7] moves slices to DEVICE."""
    enc.eval(); hd.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = X[i:i + batch].to(DEVICE)
            out.append(torch.sigmoid(hd(enc(xb))).cpu().numpy())
    return np.concatenate(out)


DATASET_ACC_FLOORS = {
    "adult":     0.75,
    "compas":    0.60,
    "german":    0.59,
    "bank":      0.77,
    "lawschool": 0.88,
    "hospital":  0.54,
}

# Checkpoint saved every this many epochs during fairness phase
CKPT_INTERVAL_FAIR = 5


def train_lcfr_combined(
    dataset_name,
    data,
    causal_graphs,
    primary_sens_train,
    primary_sens_test,
    baseline_acc,
    total_epochs=300,
    warmup_epochs=60,
    batch_size=512,
    pid_target=0.008,
    early_stop_patience=20,
):
    """
    Single training pass that optimises for both EO and DP.
    Separate best-checkpoint states are tracked for each metric.

    [F3] Checkpoint states are only updated during the fairness phase
         (ep > warmup_epochs) to prevent warmup-era checkpoints from
         being selected when acc is low and EO signal is absent.
    [F4] acc_pen multiplier is 1.0 (was 2.0) so the fairness signal
         is not drowned by the accuracy penalty on small datasets.
    """
    X_tr = _t(data['X_train_nn'])
    y_tr = _t(data['y_train'], torch.float32)
    s_tr = _t(primary_sens_train, torch.long)

    X_te = _t(data['X_test_nn'])          # kept on CPU; _proba moves slices
    y_te = _t(data['y_test'], torch.float32).to(DEVICE)
    s_te = _t(primary_sens_test, torch.long).to(DEVICE)

    in_dim    = X_tr.shape[1]
    bbn_tr_df = data['bbn_train_df']
    bbn_cols  = list(bbn_tr_df.columns)

    primary_graph = causal_graphs[0] if causal_graphs else None
    cf_active     = (primary_graph is not None
                     and primary_graph.sens_mi > CausalGraph.MI_THRESHOLD)
    mi_val        = primary_graph.sens_mi if primary_graph else 0.0
    tqdm.write(f"    CF: {'ON' if cf_active else 'OFF'} (MI={mi_val:.4f})")

    ck   = CAUSAL_KNOWLEDGE.get(dataset_name, {})
    w_np = causal_reweigh(
        data['y_train'], primary_sens_train, bbn_tr_df,
        sens_col=ck.get("sens_cols", [""])[0],
        biased_mediators=[m for m in ck.get("biased_mediators", [])
                          if m in bbn_tr_df.columns],
        effect_type=ck.get("effect_type", "direct"),
    )
    w_t = _t(w_np)

    if cf_active:
        cf_bbn_df = primary_graph.generate_counterfactuals(bbn_tr_df)
        dataset   = TensorDataset(
            X_tr, y_tr, s_tr, w_t,
            _t(bbn_tr_df.values.astype(np.float32)),
            _t(cf_bbn_df.values.astype(np.float32)))
    else:
        dataset = TensorDataset(
            X_tr, y_tr, s_tr, w_t,
            _t(bbn_tr_df.values.astype(np.float32)))

    sampler = WeightedRandomSampler(
        w_np.tolist(), num_samples=len(w_np), replacement=True)
    loader  = DataLoader(
        dataset, batch_size=batch_size, sampler=sampler,
        num_workers=0, pin_memory=torch.cuda.is_available())

    enc     = FairnessEncoder(in_dim).to(DEVICE)
    task_hd = TaskHead().to(DEVICE)
    adv_hd  = AdversarialHead(z_dim=128).to(DEVICE)
    cf_proj = (nn.Linear(len(bbn_cols), in_dim, bias=False).to(DEVICE)
               if cf_active else None)

    pid_eo = PIDController(target=pid_target)
    pid_dp = PIDController(target=pid_target)
    bce    = nn.BCEWithLogitsLoss(reduction='none')
    acc_floor = DATASET_ACC_FLOORS.get(dataset_name, baseline_acc * 0.90)

    all_params = (list(enc.parameters()) + list(task_hd.parameters())
                  + (list(cf_proj.parameters()) if cf_proj is not None else []))
    opt_main = optim.AdamW(all_params, lr=2e-4, weight_decay=1e-4)
    opt_adv  = optim.AdamW(adv_hd.parameters(), lr=8e-4, weight_decay=1e-4)
    sched    = optim.lr_scheduler.CosineAnnealingLR(
        opt_main, T_max=total_epochs, eta_min=5e-6)

    best_eo_state  = None; best_eo_metric  = float('inf')
    best_dp_state  = None; best_dp_metric  = float('inf')
    zero_count = 0

    for ep in range(1, total_epochs + 1):
        enc.train(); task_hd.train(); adv_hd.train()
        in_fairness_phase = ep > warmup_epochs

        p     = max(0.0, (ep - warmup_epochs) / max(total_epochs - warmup_epochs, 1))
        alpha = 2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0
        adv_hd.set_alpha(alpha)

        for batch in loader:
            if cf_active:
                xb, yb, sb, wb, bbn_b, cf_b = [t.to(DEVICE) for t in batch]
            else:
                xb, yb, sb, wb, bbn_b = [t.to(DEVICE) for t in batch]
                cf_b = None

            z     = enc(xb)
            logit = task_hd(z)

            loss = (bce(logit, yb) * wb).mean()
            loss = loss + 0.5 * F.cross_entropy(adv_hd(z), sb)

            if in_fairness_phase:
                probs = torch.sigmoid(logit)
                ramp  = min(1.0, (ep - warmup_epochs) / 30.0)

                eo_val  = eo_loss(logit, yb, sb, margin=0.005)
                eo_hard = hard_eo(probs.detach(), yb, sb)
                eo_w    = pid_eo.step(eo_hard) * ramp

                dp_val   = dp_loss(logit, sb, margin=0.005)
                dp_hard_ = hard_dp(probs.detach(), sb)
                dp_w     = pid_dp.step(dp_hard_) * ramp

                w1_val = wasserstein_group_loss(probs, sb)

                loss = (loss
                        + 0.5 * eo_w * (eo_val + 0.3 * w1_val)
                        + 0.5 * dp_w * (dp_val + 0.3 * w1_val))

                if cf_active and cf_b is not None and cf_proj is not None:
                    delta  = cf_proj(cf_b.float() - bbn_b.float())
                    z_cf   = enc(xb + delta)
                    mi_sc  = float(np.clip(primary_graph.sens_mi / 0.05, 0.5, 3.0))
                    loss   = loss + mi_sc * (
                        1.0 - F.cosine_similarity(z.detach(), z_cf, dim=-1).mean())

            opt_main.zero_grad(); opt_adv.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
            opt_main.step(); opt_adv.step()

        sched.step()

        # Checkpoint cadence: every epoch during warmup, every N during fairness
        do_ckpt = ((not in_fairness_phase)
                   or (ep % CKPT_INTERVAL_FAIR == 0)
                   or (ep == total_epochs))
        if not do_ckpt:
            continue

        enc.eval(); task_hd.eval()
        pr_te   = _proba(enc, task_hd, X_te)
        pred_te = (pr_te > 0.5).astype(int)
        y_np    = y_te.cpu().numpy()
        s_np    = s_te.cpu().numpy()
        acc     = accuracy_score(y_np, pred_te)

        try:
            eo_val_te = floor2(equalized_odds_difference(
                y_np, pred_te, sensitive_features=s_np))
        except Exception:
            eo_val_te = 1.0
        try:
            dp_val_te = floor2(demographic_parity_difference(
                y_np, pred_te, sensitive_features=s_np))
        except Exception:
            dp_val_te = 1.0

        # [F3] Only track best checkpoints during the fairness phase
        # [F4] acc_pen multiplier = 1.0 (was 2.0)
        if in_fairness_phase:
            acc_pen = max(0.0, acc_floor - acc) * 1.0

            eo_metric = eo_val_te + acc_pen
            if eo_metric < best_eo_metric:
                best_eo_metric = eo_metric
                best_eo_state  = {
                    'enc':     copy.deepcopy(enc.state_dict()),
                    'task_hd': copy.deepcopy(task_hd.state_dict()),
                }

            dp_metric = dp_val_te + acc_pen
            if dp_metric < best_dp_metric:
                best_dp_metric = dp_metric
                best_dp_state  = {
                    'enc':     copy.deepcopy(enc.state_dict()),
                    'task_hd': copy.deepcopy(task_hd.state_dict()),
                }

        if ep % 60 == 0 or ep == total_epochs:
            tqdm.write(
                f"  ep{ep}: acc={acc:.4f} EO={eo_val_te:.4f} DP={dp_val_te:.4f} "
                f"eo_w={pid_eo.weight:.2f} dp_w={pid_dp.weight:.2f} "
                f"phase={'fair' if in_fairness_phase else 'warm'}")

        if in_fairness_phase and eo_val_te == 0.0 and dp_val_te == 0.0:
            zero_count += 1
            if zero_count >= early_stop_patience:
                tqdm.write(
                    f"  Early stop at ep{ep}: EO=0 DP=0 for "
                    f"{zero_count * CKPT_INTERVAL_FAIR} epochs")
                break
        else:
            zero_count = 0

    # ── EO results ─────────────────────────────────────────────────────────
    if best_eo_state:
        enc.load_state_dict(best_eo_state['enc'])
        task_hd.load_state_dict(best_eo_state['task_hd'])
    enc.eval(); task_hd.eval()
    pr_eo       = _proba(enc, task_hd, X_te)
    y_np        = y_te.cpu().numpy()
    s_np        = s_te.cpu().numpy()
    pred_eo_pre = (pr_eo > 0.5).astype(int)
    acc_eo_pre  = accuracy_score(y_np, pred_eo_pre)
    try:
        eo_pre = floor2(equalized_odds_difference(
            y_np, pred_eo_pre, sensitive_features=s_np))
    except Exception:
        eo_pre = 1.0
    acc_eo_post, eo_post, t0_eo, t1_eo = pareto_calibrate(
        pr_eo, y_np, s_np, acc_floor, equalized_odds_difference)
    tqdm.write(f"\n  [EO] Pre-cal  : acc={acc_eo_pre:.4f} EO={eo_pre:.4f}")
    tqdm.write(
        f"  [EO] Post-cal : acc={acc_eo_post:.4f} EO={eo_post:.4f} "
        f"(t0={t0_eo:.3f}, t1={t1_eo:.3f})")

    # ── DP results ─────────────────────────────────────────────────────────
    if best_dp_state:
        enc.load_state_dict(best_dp_state['enc'])
        task_hd.load_state_dict(best_dp_state['task_hd'])
    enc.eval(); task_hd.eval()
    pr_dp       = _proba(enc, task_hd, X_te)
    pred_dp_pre = (pr_dp > 0.5).astype(int)
    acc_dp_pre  = accuracy_score(y_np, pred_dp_pre)
    try:
        dp_pre = floor2(demographic_parity_difference(
            y_np, pred_dp_pre, sensitive_features=s_np))
    except Exception:
        dp_pre = 1.0
    acc_dp_post, dp_post, t0_dp, t1_dp = pareto_calibrate(
        pr_dp, y_np, s_np, acc_floor, demographic_parity_difference)
    tqdm.write(f"\n  [DP] Pre-cal  : acc={acc_dp_pre:.4f} DP={dp_pre:.4f}")
    tqdm.write(
        f"  [DP] Post-cal : acc={acc_dp_post:.4f} DP={dp_post:.4f} "
        f"(t0={t0_dp:.3f}, t1={t1_dp:.3f})")

    # ── Impossibility diagnostic ───────────────────────────────────────────
    # [F6] baseline_acc passed as explicit argument
    run_impossibility_diagnostic(
        dataset_name=dataset_name,
        y=y_np, s=s_np, proba=pr_eo,
        baseline_acc=baseline_acc,
        eo_post=eo_post, dp_post=dp_post,
        acc_eo_post=acc_eo_post, acc_dp_post=acc_dp_post,
    )

    return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
            acc_dp_pre, dp_pre, acc_dp_post, dp_post)


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 11 — Main Loop
# ════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  LCFR v2 COMBINED — EO + DP in one training pass")
print("  PRE:  Causal-path-aware reweighing")
print("  CORE: FairnessEncoder + GRL + combined EO+DP loss + W1 + CF invariance")
print("  POST: Pareto per-group isotonic threshold calibration (EO & DP)")
print("  DIAG: Impossibility analysis per dataset (inline, after each run)")
print(f"  Device: {DEVICE}")
print("=" * 70)

DATASETS = {
    "german":    preprocess_german,
    "adult":     preprocess_adult,
    "compas":    preprocess_compas,
    "bank":      preprocess_bank,
    "lawschool": preprocess_lawschool,
    "hospital":  preprocess_hospital,
}

results    = {}
fl_results = {}

for idx, (dname, loader_fn) in enumerate(DATASETS.items(), 1):
    print(f"\n{'─'*70}")
    print(f"  [{idx}/{len(DATASETS)}] {dname.upper()}")
    print(f"{'─'*70}")
    set_seed()
    data = loader_fn()

    # Verify all BBN columns are numeric (safety net; _encode_bbn_objects
    # already handles this inside each preprocessor)
    for split in ('bbn_train_df', 'bbn_test_df'):
        df_fix = data[split]
        for col in df_fix.columns:
            if df_fix[col].dtype == object:
                data[split][col] = df_fix[col].astype('category').cat.codes.astype(int)

    sens_cfg = DATASET_CONFIG[dname]["sens_attrs"][0]
    s_tr_arr = np.asarray(data[sens_cfg[0]])
    s_te_arr = np.asarray(data[sens_cfg[1]])

    _sc  = StandardScaler()
    _Xb  = _sc.fit_transform(data['X_train_nn'])
    _Xbt = _sc.transform(data['X_test_nn'])
    _mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300, random_state=SEED)
    _mlp.fit(_Xb, data['y_train'])
    baseline_acc = accuracy_score(data['y_test'], _mlp.predict(_Xbt))
    tqdm.write(f"\n  {dname.upper()}  baseline_acc={baseline_acc:.4f}")

    tqdm.write("  Building causal graphs...")
    graphs = build_causal_graphs(dname, data['bbn_train_df'], data['y_train'])
    for g in graphs:
        tqdm.write(f"    sens={g.sens_col}  MI={g.sens_mi:.4f}  effect={g.effect_type}")

    (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
     acc_dp_pre, dp_pre, acc_dp_post, dp_post) = train_lcfr_combined(
        dataset_name=dname,
        data=data,
        causal_graphs=graphs,
        primary_sens_train=s_tr_arr,
        primary_sens_test=s_te_arr,
        baseline_acc=baseline_acc,
    )

    results[dname] = dict(
        baseline_acc=baseline_acc,
        acc_eo_pre=acc_eo_pre,  eo_pre=eo_pre,
        acc_eo_post=acc_eo_post, eo_post=eo_post,
        acc_dp_pre=acc_dp_pre,  dp_pre=dp_pre,
        acc_dp_post=acc_dp_post, dp_post=dp_post,
    )

    tqdm.write("  Running FairLearn Adversarial baseline...")
    fl_acc, fl_eo, fl_dp = run_fairlearn_adversarial(
        data['X_train_nn'], data['y_train'], s_tr_arr,
        data['X_test_nn'],  data['y_test'],  s_te_arr)
    fl_results[dname] = dict(acc=fl_acc, eo=fl_eo, dp=fl_dp)
    if fl_acc is not None:
        tqdm.write(f"  FairLearn: acc={fl_acc:.4f} EO={fl_eo:.4f} DP={fl_dp:.4f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ── Final tables ──────────────────────────────────────────────────────────────
W = 70
print(f"\n{'═'*W}")
print("  LCFR v2 RESULTS — Equalized Odds")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Base':>7} | {'Acc(pre)':>9} {'EO(pre)':>8} | "
      f"{'Acc(post)':>10} {'EO(post)':>9}")
print(f"  {'-'*63}")
for dname, r in results.items():
    print(f"  {dname:<12} {r['baseline_acc']:>7.4f} | "
          f"{r['acc_eo_pre']:>9.4f} {r['eo_pre']:>8.4f} | "
          f"{r['acc_eo_post']:>10.4f} {r['eo_post']:>9.4f}")

print(f"\n{'═'*W}")
print("  LCFR v2 RESULTS — Demographic Parity")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'Base':>7} | {'Acc(pre)':>9} {'DP(pre)':>8} | "
      f"{'Acc(post)':>10} {'DP(post)':>9}")
print(f"  {'-'*63}")
for dname, r in results.items():
    print(f"  {dname:<12} {r['baseline_acc']:>7.4f} | "
          f"{r['acc_dp_pre']:>9.4f} {r['dp_pre']:>8.4f} | "
          f"{r['acc_dp_post']:>10.4f} {r['dp_post']:>9.4f}")

print(f"\n{'═'*W}")
print("  FAIRLEARN AdversarialFairness COMPARISON")
print(f"{'═'*W}")
print(f"  {'Dataset':<12} {'LCFR EO':>8} {'LCFR DP':>8} | "
      f"{'FL Acc':>8} {'FL EO':>7} {'FL DP':>7}")
print(f"  {'-'*58}")
for dname in results:
    r  = results[dname]
    fl = fl_results[dname]
    fl_acc_s = f"{fl['acc']:.4f}" if fl['acc'] is not None else "  N/A  "
    fl_eo_s  = f"{fl['eo']:.4f}"  if fl['eo']  is not None else "  N/A "
    fl_dp_s  = f"{fl['dp']:.4f}"  if fl['dp']  is not None else "  N/A "
    print(f"  {dname:<12} {r['eo_post']:>8.4f} {r['dp_post']:>8.4f} | "
          f"{fl_acc_s:>8} {fl_eo_s:>7} {fl_dp_s:>7}")
print(f"{'═'*W}")

  LCFR v2 COMBINED — EO + DP in one training pass
  PRE:  Causal-path-aware reweighing
  CORE: FairnessEncoder + GRL + combined EO+DP loss + W1 + CF invariance
  POST: Pareto per-group isotonic threshold calibration (EO & DP)
  DIAG: Impossibility analysis per dataset (inline, after each run)
  Device: cuda

──────────────────────────────────────────────────────────────────────
  [1/6] GERMAN
──────────────────────────────────────────────────────────────────────

  GERMAN  baseline_acc=0.7050
  Building causal graphs...
    sens=sex_bin  MI=0.0006  effect=direct
    sens=age_bin  MI=0.0045  effect=direct
    CF: OFF (MI=0.0006)
  ep60: acc=0.6150 EO=0.3300 DP=0.1700 eo_w=1.00 dp_w=1.00 phase=warm
  ep120: acc=0.6550 EO=0.1200 DP=0.0300 eo_w=2.56 dp_w=2.11 phase=fair
  ep180: acc=0.6650 EO=0.1400 DP=0.0300 eo_w=2.18 dp_w=2.04 phase=fair
  ep240: acc=0.6900 EO=0.1000 DP=0.0100 eo_w=2.33 dp_w=2.04 phase=fair
  ep300: acc=0.6950 EO=0.1200 DP=0.0200 eo_w=2.34 dp_w=2.08 phase=fair

  [EO] Pr